# Milestone 5 lab, notebook 7 of 8: Four drills

## Where we are

| Notebook | What you do |
|---|---|
| 1. Did it run? | Read Airflow's run history and the pipeline's own. Run one night twice |
| 2. On time, as a number | Measure freshness. Turn "on time" into a number with a target |
| 3. Five signals | Monitoring asks if the job ran. Observability asks if the data is right |
| 4. Tell the person who can act | Airflow calls back, and the right person gets an alert |
| 5. Too many alerts | Count the careless way. Then group, send once, rank and silence |
| 6. Fix one day | Fix the source, run one night again, prove the number |
| **7. Four drills** (you are here) | Four faults on a clean day, each caught and repaired. Write an incident note |
| 8. The Monday answer | The whole week, the recovery, and the answer to the question |

**You need:** the warehouse and Airflow running. From the project folder: `docker compose up -d --build`. Airflow's page is http://localhost:8080 .

**About ten minutes of reading, and four minutes of waiting.** Each drill starts from nothing and lets Airflow run a night or two.

In [1]:
from datetime import date

import pandas as pd
from lab_helpers import (
    DRILL_DAY, INCIDENTS_DIR, MILESTONE_DIR, airflow_tools, alerts_of, backfill, gate_results,
    health_numbers, manager_line, money, proof, set_aside, sql, start_drill,
)

airflow_tools.wait_until_ready()

## Step 1: A fire drill, for data

**Aiko Tanaka** runs store `S06`. In this notebook her store will send a file in which 12 prices are a hundred times too high. The pipeline must stop it before it reaches her bonus.

A fire drill does not wait for a fire. You cause four faults on purpose, one at a time, and practise the whole recovery. Each drill runs on Thursday 20 August, a clean day, so it stands alone.

| Drill | The fault | Store |
|---|---|---|
| 1 | The store's whole file arrives twice | `S03` |
| 2 | A column of the file is renamed | `S04` |
| 3 | The file arrives a day late | `S05` |
| 4 | A batch of impossible prices | `S06` |

For each drill, six questions: which gate caught it, what was set aside, what the manager saw, who was told, the fix, and the proof against the answer key.

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, what happens when it fails, and who is told.

## Drill 1: A file arrives twice

`start_drill` empties everything, switches the fault on at the store, and lets Airflow run that night.

In [2]:
start_drill("file-twice", "S03")

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.
Next time S03 sends 2026-08-20: file-twice. Deliver that day to see it.


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [3]:
gate_results(DRILL_DAY)

,gate,test_name,action,status,bad_rows,title
0,clean,true_key_unique,quarantine,fail,121,One item line is stored once
1,clean,normal_volume,warn,fail,1,A store's day is near its normal day


In [4]:
set_aside("S03", DRILL_DAY)

,why,lines,cents
0,true_key_unique,121,71282


**Caught by** the clean gate: "one item line is stored once". The second file had new line ids, so the line id is still unique, but the same receipt line is there twice. **Set aside**: 121 lines, every line of the second file, as extra copies. The volume test also warned: the day is far from normal.

In [5]:
manager_line("S03", DRILL_DAY)

S03  Tomas Reyes  net sales $2,978.76  Pay? yes, bonus $29.79


In [6]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,message,Head of retail operations,S03 Thu 08-20 had a day far from its normal day
1,new,message,The data engineer on call,S03 Thu 08-20 sent sales lines twice. The extra copies were removed


In [7]:
proof("S03", DRILL_DAY)

,store,day,published,answer key,equal,status
0,S03,Thu 08-20,$712.82,$712.82,True,ok


**The manager saw** the right number, and "yes". **Who was told:** nobody. Two messages wait in the digest. Nobody needs to act tonight.

**The fix:** none. This is the first honest surprise. A store cannot take a file back: the second copy stays in the landing folder for good. So every night the gate sets its lines aside again. Run the night once more and see:

In [8]:
pd.DataFrame(airflow_tools.run_night(DRILL_DAY.isoformat()))

,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [9]:
display(set_aside("S03", DRILL_DAY))
proof("S03", DRILL_DAY)

,why,lines,cents
0,true_key_unique,121,71282


,store,day,published,answer key,equal,status
0,S03,Thu 08-20,$712.82,$712.82,True,ok


The same 121 lines set aside, and the same right number. **The proof** equals the answer key.

## Drill 2: A column is renamed

The store's export program now calls the column `amount_cents` just `amount`.

In [10]:
start_drill("rename-column", "S04")

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.
Next time S04 sends 2026-08-20: rename-column. Deliver that day to see it.


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [11]:
gate_results(DRILL_DAY)

,gate,test_name,action,status,bad_rows,title
0,landing,file_is_missing_a_column,refuse_file,fail,1,A file has every column we expect
1,landing,file_has_a_new_column,warn,fail,1,A file has no column we did not expect
2,publish,store_is_not_ready_to_pay,warn,fail,1,Every open store has a day we can pay from


In [12]:
sql("SELECT file_name, reason FROM ops.refused_files")

,file_name,reason
0,sales_lines_S04_2026-08-20.csv,An expected column is missing: amount_cents


**Caught by** the landing gate: "a file has every column we expect". **Set aside:** the whole file, refused. Without `amount_cents`, not one of its rows has a price.

In [13]:
manager_line("S04", DRILL_DAY)

S04  Ines Marchetti  net sales $2,473.14  Pay? DO NOT PAY YET
     Thu 08-20: the sales file was refused. An expected column is missing: amount_cents


In [14]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,ticket,Head of retail operations,S04 Thu 08-20: The sales file was refused
1,new,message,The data engineer on call,sales_lines_S04_2026-08-20.csv has a column nobody agreed


**The manager saw** "do not pay yet", with the reason. **Who was told:** a ticket to the head of retail operations, and a message for the engineer on call about the new column.

**The fix:** the store sends a corrected file. Here that is one command that switches the fault off. Then a backfill of that one night.

In [15]:
print(airflow_tools.stores("fix", "--all-faults"))
pd.DataFrame(backfill("2026-08-20", "2026-08-20"))

Deliver the day again to get the corrected files.


,day,state
0,2026-08-20,success


In [16]:
proof("S04", DRILL_DAY)

,store,day,published,answer key,equal,status
0,S04,Thu 08-20,$575.71,$575.71,True,ok


In [17]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,ticket,Head of retail operations,S04 Thu 08-20: The sales file was refused
1,new,message,The data engineer on call,sales_lines_S04_2026-08-20.csv has a column nobody agreed
2,resolved,ticket,Head of retail operations,S04 Thu 08-20: The sales file was refused
3,resolved,message,The data engineer on call,sales_lines_S04_2026-08-20.csv has a column nobody agreed


**The proof:** equal to the answer key, status `ok`. Both alerts are `resolved`.

## Drill 3: A file is a day late

`S05`'s Thursday file will come during Friday night, at 00:40.

In [18]:
start_drill("late-file", "S05")

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.
Next time S05 sends 2026-08-20: late-file. Deliver that day to see it.


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [19]:
gate_results(DRILL_DAY)

,gate,test_name,action,status,bad_rows,title
0,publish,store_is_not_ready_to_pay,warn,fail,1,Every open store has a day we can pay from


In [20]:
manager_line("S05", DRILL_DAY)

S05  Samir Haddad  net sales $1,393.05  Pay? DO NOT PAY YET
     Thu 08-20: no sales file arrived. The tills counted $578.65


In [21]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,ticket,Head of retail operations,S05 Thu 08-20: No sales file arrived


**Caught by** the publish gate: "every open store has a day we can pay from". **Set aside:** nothing. There was nothing to set aside. **The manager saw** "no sales file arrived". **Who was told:** a ticket to the head of retail operations.

**The fix:** none. This is the second honest surprise. The file is on its way. The next night loads it by itself. Run Friday's night:

In [22]:
pd.DataFrame(airflow_tools.run_night("2026-08-21"))

,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [23]:
friday = gate_results(date(2026, 8, 21))
display(friday[friday["test_name"] == "file_arrived_on_time"])
print("Late files:", health_numbers(date(2026, 8, 21))["late_files"])

,gate,test_name,action,status,bad_rows,title
0,landing,file_arrived_on_time,warn,fail,1,A store's file arrives before 01:00


Late files: [{'store_id': 'S05', 'day': '2026-08-20', 'hours_late': 23.7}]


In [24]:
display(proof("S05", DRILL_DAY))
alerts_of(date(2026, 8, 21), about="S05")

,store,day,published,answer key,equal,status
0,S05,Thu 08-20,$578.65,$578.65,True,ok


,event,level,to_whom,title
0,new,message,Store operations,S05 Thu 08-20 sent its file late
1,resolved,ticket,Head of retail operations,S05 Thu 08-20: No sales file arrived


Freshness names it: 23.7 hours late. **The proof:** equal to the answer key. The ticket was resolved on Friday night, and nobody did anything. The message about the late file waits in the digest.

## Drill 4: A batch of impossible prices

Back to Aiko. Her store's export now writes some prices a hundred times too high.

In [25]:
start_drill("bad-prices", "S06")

Empty: the warehouse of this milestone, the stores' landing folder, the alerts log,
and Airflow's runs of lotus_nightly_m5.
Next time S06 sends 2026-08-20: bad-prices. Deliver that day to see it.


,task,state
0,stores_send_files,success
1,load,success
2,gate_landing,success
3,clean,success
4,gate_clean,success
5,build,success
6,gate_publish,success
7,publish,success


In [26]:
gate_results(DRILL_DAY)

,gate,test_name,action,status,bad_rows,title
0,clean,line_adds_up,quarantine,fail,2,A line adds up
1,clean,price_of_the_day,warn,fail,12,The price charged is the price valid that day
2,clean,impossible_price,quarantine,fail,12,A price is not impossible
3,publish,store_is_not_ready_to_pay,warn,fail,1,Every open store has a day we can pay from


In [27]:
aside = set_aside("S06", DRILL_DAY)
aside

,why,lines,cents
0,line_adds_up,2,25100
1,impossible_price,12,630000
2,too_many_bad_rows,106,71559


**Caught by** the clean gate: "a price is not impossible", a new test of this milestone. It sets aside a price more than twice the price list, or less than half of it. Read the rows of `aside`:

- `impossible_price`: 12 lines, for $6,300.00.
- `line_adds_up`: 2 lines whose amount is not quantity times price.
- `too_many_bad_rows`: then the whole file goes. More than 5% of it was bad, and when that much of a file is broken, the rest is not trusted either. Milestone 4 set that rule.

In [28]:
manager_line("S06", DRILL_DAY)

S06  Aiko Tanaka  net sales $1,497.56  Pay? DO NOT PAY YET
     Thu 08-20: 118 rows set aside ($7,015.59): A line adds up; A price is not impossible; More than 5% of this file was set aside, so the whole file is refused


In [29]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,ticket,Finance,S06 Thu 08-20 charged a price that is not in the price list
1,new,ticket,Head of retail operations,"S06 Thu 08-20: Rows were set aside, and the day does not add up"


**The manager saw** "do not pay yet", with every reason. **Who was told:** two tickets. The one to the head of retail operations is the one that matters.

The second, to finance, comes from an older warning: "the price charged is the price valid that day". Its message says the sales number is right, which is not true tonight. Two tests saw the same rows, and two people were told. That is the kind of thing a drill is for: you find it on a Thursday you chose, not on a real night.

**The fix:** a corrected file, and a backfill of that night.

In [30]:
print(airflow_tools.stores("fix", "--all-faults"))
pd.DataFrame(backfill("2026-08-20", "2026-08-20"))

Deliver the day again to get the corrected files.


,day,state
0,2026-08-20,success


In [31]:
after = proof("S06", DRILL_DAY)
after

,store,day,published,answer key,equal,status
0,S06,Thu 08-20,$778.59,$778.59,True,ok


In [32]:
alerts_of(DRILL_DAY)

,event,level,to_whom,title
0,new,ticket,Finance,S06 Thu 08-20 charged a price that is not in the price list
1,new,ticket,Head of retail operations,"S06 Thu 08-20: Rows were set aside, and the day does not add up"
2,resolved,ticket,Finance,S06 Thu 08-20 charged a price that is not in the price list
3,resolved,ticket,Head of retail operations,"S06 Thu 08-20: Rows were set aside, and the day does not add up"


**The proof:** equal to the answer key, status `ok`. Both tickets `resolved`.

## Step 2: The four drills, side by side

| Drill | Caught by | Set aside | The manager saw | Who was told | The fix |
|---|---|---|---|---|---|
| File twice, `S03` | Clean gate: one item line is stored once | 121 copies | The right number, "yes" | Nobody. Two messages in the digest | None. The gate removes the copies every night |
| Column renamed, `S04` | Landing gate: a file has every column | The whole file, refused | "Do not pay yet: the sales file was refused" | A ticket | A corrected file, and a backfill of one night |
| File a day late, `S05` | Publish gate: every open store has a day we can pay from | Nothing | "Do not pay yet: no sales file arrived" | A ticket, resolved the next night | None. It arrives, and freshness names it: 23.7 hours late |
| Impossible prices, `S06` | Clean gate: a price is not impossible | 12 lines, then the whole file | "Do not pay yet", with the rows | Two tickets | A corrected file, and a backfill of one night |

After each drill the published number equalled the answer key.

## Step 3: An incident note

An **incident note** is a short written record of an incident: what happened, who was hurt, why, and what was changed. It is also called a **post-mortem**. It blames nobody. The next person learns from it without having been there.

Its opposite is the incident everyone remembers differently a month later.

This project keeps a template in `incidents/TEMPLATE.md`. Fill it in for drill 4, from the numbers above, and save it.

In [33]:
impossible = aside[aside["why"] == "impossible_price"].iloc[0]
whole_file = aside[aside["why"] == "too_many_bad_rows"].iloc[0]
lines_in_file = int(sql(
    "SELECT count(*) AS n FROM raw.sales_lines WHERE _file_name = 'sales_lines_S06_2026-08-20.csv'"
).loc[0, "n"])
right = after.iloc[0]

note = f"""# Incident note: impossible prices at S06 on Thursday 2026-08-20

An incident note is a short written record of an incident. It is also called a post-mortem. It blames nobody.
This one was written in the milestone 5 lab, notebook 7, after a drill: the fault was planted on purpose.

| | |
|---|---|
| **Day of the data** | Thursday 2026-08-20 |
| **Found** | The run of that night, 02:00 on 2026-08-21, by the clean gate: the test `impossible_price` |
| **Fixed** | The same day, by a backfill of the night 2026-08-20 |
| **Written by** | The Lotus data team, in the lab |

## What happened

Store S06 sent its sales file for Thursday 2026-08-20. In {int(impossible["lines"])} of its {lines_in_file} lines the price charged was a hundred times the price list.
Those {int(impossible["lines"])} lines alone came to {money(impossible["cents"])}. The answer key, added up from the receipts, says the whole day was {right["answer key"]}.

## Who was hurt, and how much

Aiko Tanaka, the manager of S06. Her day was marked "do not pay yet" for one night, so her bonus waited.
Nobody saw a wrong number: the bad lines never reached the published tables.

## How we found out

The clean gate. The test `impossible_price` set {int(impossible["lines"])} lines aside. Because more than 5% of the file was bad,
the whole file was then set aside: {int(whole_file["lines"])} more lines, with the reason `too_many_bad_rows`.
The head of retail operations got a ticket on the night. Finance got a second ticket from the warning `price_of_the_day`.
No person noticed before the pipeline did.

## Root cause

The store's export program wrote some prices in the wrong unit, a hundred times too large.
The check is how we found out. The export is what went wrong.

## What we did

1. The store sent a corrected file under the same name.
2. We ran only the night of 2026-08-20 again, with an Airflow backfill. The load step saw a new fingerprint and loaded the file again.
3. Proof: S06 on 2026-08-20 is published as {right["published"]}. The answer key says {right["answer key"]}. The status is `{right["status"]}`. Both tickets were resolved.

## What we changed so it cannot happen silently again

Nothing needed to change in the checks: the gate caught it on the night, and that is a good result.
One thing to improve: the warning `price_of_the_day` also sent a ticket to finance, saying the sales number was right.
For a price that is impossible, that ticket is noise. A test that already sets a row aside should silence the warning on the same row.
"""

path = INCIDENTS_DIR / "2026-08-20-impossible-prices-at-S06.md"
path.write_text(note)
print("Saved", path.relative_to(MILESTONE_DIR))
print()
print(note)

Saved incidents/2026-08-20-impossible-prices-at-S06.md

# Incident note: impossible prices at S06 on Thursday 2026-08-20

An incident note is a short written record of an incident. It is also called a post-mortem. It blames nobody.
This one was written in the milestone 5 lab, notebook 7, after a drill: the fault was planted on purpose.

| | |
|---|---|
| **Day of the data** | Thursday 2026-08-20 |
| **Found** | The run of that night, 02:00 on 2026-08-21, by the clean gate: the test `impossible_price` |
| **Fixed** | The same day, by a backfill of the night 2026-08-20 |
| **Written by** | The Lotus data team, in the lab |

## What happened

Store S06 sent its sales file for Thursday 2026-08-20. In 12 of its 118 lines the price charged was a hundred times the price list.
Those 12 lines alone came to $6,300.00. The answer key, added up from the receipts, says the whole day was $778.59.

## Who was hurt, and how much

Aiko Tanaka, the manager of S06. Her day was marked "do not pay yet" for

One page, and every number in it came from this notebook.

## What you now know

- A **drill** causes a fault on purpose, on a day you choose, and practises the whole recovery.
- Four faults, four gates: a file twice at the clean gate, a renamed column at the landing gate, a late file at the publish gate, impossible prices at the clean gate.
- Two faults needed no fix. A file sent twice cannot be taken back, so the gate removes the copies every night. A late file heals by itself the next night.
- After every fix, the number equalled the answer key.
- An **incident note**, also called a post-mortem, says what happened, who was hurt, how it was found, the root cause, what was done, and what was changed.

**Say it in an interview:** "I ran four drills on a clean day: a file sent twice, a renamed column, a late file and a batch of impossible prices. Each was caught at a gate, and after each fix the number equalled the answer key. Two needed no fix at all. After each drill I wrote a short incident note: what happened, who was hurt and by how much, the root cause, what we did, and what we changed. The impossible-prices drill found a second ticket that was noise, so the note says to silence it."

**The theory:** [Incremental Processing & Backfills](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/incremental-processing-and-backfills), and [Data Observability & Monitoring](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-observability-and-monitoring), section on incidents.

Next: [notebook 8](08_the_monday_answer.ipynb). The whole week, one last time.